In [1]:
import pandas as pd

df = pd.read_csv(r"C:\Users\death\MyPythonProjects\PythonProject1\ecommerce behavior data from multi category store\2019-Nov.csv", sep = ",")


print(df.shape)
df.head(5)

(67501979, 9)


,event_time,event_type,product_id,category_id,category_code,brand,price,user_id,user_session
0,2019-11-01 00:00:00 UTC,view,1003461,2053013555631882655,electronics.smartphone,xiaomi,489.07,520088904,4d3b30da-a5e4-49df-b1a8-ba5943f1dd33
1,2019-11-01 00:00:00 UTC,view,5000088,2053013566100866035,appliances.sewing_machine,janome,293.65,530496790,8e5f4f83-366c-4f70-860e-ca7417414283
2,2019-11-01 00:00:01 UTC,view,17302664,2053013553853497655,NaN,creed,28.31,561587266,755422e7-9040-477b-9bd2-6a6e8fd97387
3,2019-11-01 00:00:01 UTC,view,3601530,2053013563810775923,appliances.kitchen.washer,lg,712.87,518085591,3bfb58cd-7892-48cc-8020-2f17e6de6e7f
4,2019-11-01 00:00:01 UTC,view,1004775,2053013555631882655,electronics.smartphone,xiaomi,183.27,558856683,313628f1-68b8-460d-84f6-cec7a8796ef2


In [2]:
df.columns

Index(['event_time', 'event_type', 'product_id', 'category_id',
       'category_code', 'brand', 'price', 'user_id', 'user_session'],
      dtype='object')

In [3]:
df["event_type"].value_counts()

event_type
view        63556110
cart         3028930
purchase      916939
Name: count, dtype: int64

In [4]:
df_small = df.sample(n = 200000, random_state = 42).copy()
df_small.shape

(200000, 9)

In [5]:
user_product = pd.crosstab(
    index = [df_small["user_id"], df_small["product_id"]],
    columns = df_small["event_type"]
)

user_product.head()

,event_type,cart,purchase,view
user_id,product_id,,,
110760953,100011674,0,0,1
192078182,3701005,1,0,0
210089363,16700262,0,0,1
213763705,3701340,0,0,1
249526511,12701623,0,0,1


In [6]:
user_product = user_product.reset_index()

In [7]:
if "purchase" not in user_product.columns:
    user_product["purchase"] = 0

if "view" not in user_product.columns:
    user_product["view"] = 0

if "cart" not in user_product.columns:
    user_product["cart"] = 0

user_product["label"] = (user_product["purchase"] > 0).astype(int)

print(user_product.head())
print(user_product["label"].value_counts())
print(user_product["label"].value_counts(normalize = True))

event_type    user_id  product_id  cart  purchase  view  label
0           110760953   100011674     0         0     1      0
1           192078182     3701005     1         0     0      0
2           210089363    16700262     0         0     1      0
3           213763705     3701340     0         0     1      0
4           249526511    12701623     0         0     1      0
label
0    196067
1      2669
Name: count, dtype: int64
label
0    0.98657
1    0.01343
Name: proportion, dtype: float64


In [8]:
X = user_product[["view", "cart"]]
y = user_product["label"]

print(X.head())
print("-" * 50)
print(y.head())

event_type  view  cart
0              1     0
1              0     1
2              1     0
3              1     0
4              1     0
--------------------------------------------------
0    0
1    0
2    0
3    0
4    0
Name: label, dtype: int64


In [9]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify = y
)

print(X_train.shape, X_test.shape)
print("-" * 50)
print(y_train.value_counts(normalize=True))
print("-" * 50)
print(y_test.value_counts(normalize=True))

(158988, 2) (39748, 2)
--------------------------------------------------
label
0    0.986571
1    0.013429
Name: proportion, dtype: float64
--------------------------------------------------
label
0    0.986565
1    0.013435
Name: proportion, dtype: float64


### <font color = "blue">LogisticRegression</font>

In [10]:
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(class_weight = "balanced", max_iter = 1000) #class_weight解決資料不平衡。 max_iter 調整模型訓練的迭代次數
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(39748,))

[[TN  FP]\
 [FN  TP]]

In [11]:
from sklearn.metrics import classification_report, confusion_matrix

print(confusion_matrix(y_test, y_pred))
print("-" * 50)
print(classification_report(y_test, y_pred))

[[39214     0]
 [   24   510]]
--------------------------------------------------
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     39214
           1       1.00      0.96      0.98       534

    accuracy                           1.00     39748
   macro avg       1.00      0.98      0.99     39748
weighted avg       1.00      1.00      1.00     39748



### 時間切分版本 Purchase Prediction

### 先轉換event_time

In [48]:
df["event_time"]=pd.to_datetime(df["event_time"])
df["event_time"].min(), df["event_time"].max()

(Timestamp('2019-11-01 00:00:00+0000', tz='UTC'),
 Timestamp('2019-11-30 23:59:59+0000', tz='UTC'))

### 先用小樣本練習

In [49]:
df_small_time=df.sample(n=500000, random_state=42).copy()

df_small_time["event_time"]=pd.to_datetime(df_small_time["event_time"])

print(df_small_time["event_time"].min())
print(df_small_time["event_time"].max())

2019-11-01 00:00:50+00:00
2019-11-30 23:59:59+00:00


### 設定時間切分點

In [50]:
split_date="2019-11-21"

past_df=df_small_time[df_small_time["event_time"] < split_date].copy()
future_df=df_small_time[df_small_time["event_time"] >= split_date].copy()

print(past_df.shape)
print(future_df.shape)

(377016, 9)
(122984, 9)


### 用past_df建立特徵X

In [51]:
X_table = pd.crosstab(
    index=[past_df["user_id"], past_df["product_id"]],
    columns=past_df["event_type"]
).reset_index()

if "view" not in X_table.columns:
    X_table["view"]=0

if "cart" not in X_table.columns:
    X_table["cart"]=0

X_table=X_table[["user_id", "product_id", "view" , "cart"]]

X_table.head()

event_type,user_id,product_id,view,cart
0,110760953,100011674,1,0
1,149382035,17200858,1,0
2,158367814,26500606,1,0
3,197647707,28721201,1,0
4,210089363,13300174,1,0


### 用future_df建立答案y

In [52]:
future_purchase=future_df[future_df["event_type"] == "purchase"]


y_table=future_purchase.groupby(
    ["user_id", "product_id"]
).size().reset_index(name = "future_purchase_count")

y_table["label"] = 1

y_table.head()

,user_id,product_id,future_purchase_count,label
0,450121457,1200947,1,1
1,457745111,12710985,1,1
2,512369878,1306948,1,1
3,512371142,26400280,1,1
4,512372673,4803879,1,1


### 把X和y合併

In [53]:
model_df= X_table.merge(
    y_table[["user_id", "product_id", "label"]],
    on = ["user_id", "product_id"],
    how="left"
)

model_df["label"]=model_df["label"].fillna(0).astype(int)

model_df.head()

,user_id,product_id,view,cart,label
0,110760953,100011674,1,0,0
1,149382035,17200858,1,0,0
2,158367814,26500606,1,0,0
3,197647707,28721201,1,0,0
4,210089363,13300174,1,0,0


### 檢查新label分佈

In [54]:
print(model_df["label"].value_counts())
print(model_df["label"].value_counts(normalize=True))

label
0    371463
1        21
Name: count, dtype: int64
label
0    0.999943
1    0.000057
Name: proportion, dtype: float64


In [59]:
print(model_df.info())
print(model_df.head())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 371484 entries, 0 to 371483
Data columns (total 5 columns):
 #   Column      Non-Null Count   Dtype
---  ------      --------------   -----
 0   user_id     371484 non-null  int32
 1   product_id  371484 non-null  int32
 2   view        371484 non-null  int8 
 3   cart        371484 non-null  int8 
 4   label       371484 non-null  int8 
dtypes: int32(2), int8(3)
memory usage: 3.9 MB
None
     user_id  product_id  view  cart  label
0  110760953   100011674     1     0      0
1  149382035    17200858     1     0      0
2  158367814    26500606     1     0      0
3  197647707    28721201     1     0      0
4  210089363    13300174     1     0      0


In [63]:
int_cols=model_df.select_dtypes(include=["int64"]).columns

model_df[int_cols]=model_df[int_cols].apply(pd.to_numeric, downcast="integer")

# model_df[int_cols] = model_df[int_cols].apply(
#     pd.to_numeric,
#     downcast="integer"
# )

model_df

,user_id,product_id,view,cart,label
0,110760953,100011674,1,0,0
1,149382035,17200858,1,0,0
2,158367814,26500606,1,0,0
3,197647707,28721201,1,0,0
4,210089363,13300174,1,0,0
...,...,...,...,...,...
371479,574160409,1004155,1,0,0
371480,574161584,1005161,1,0,0
371481,574164272,10300835,1,0,0
371482,574165925,10300496,1,0,0


In [61]:
X=model_df[["view", "cart"]]
y=model_df["label"]

### 進行train, text切分

In [64]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42,
        stratify=y
)

print(X_train.shape)
print(y_train.shape)
print("================")
print(X_test.shape)
print(y_test.shape)

(297187, 2)
(297187,)
(74297, 2)
(74297,)


In [66]:
from sklearn.linear_model import LogisticRegression

model=LogisticRegression(
    class_weight="balanced",
    max_iter=1000
)


model.fit(X_train, y_train)

y_pred=model.predict(X_test)
y_pred

array([0, 0, 0, ..., 0, 0, 0], shape=(74297,), dtype=int8)

In [67]:
from sklearn.metrics import confusion_matrix, classification_report

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

[[69969  4324]
 [    3     1]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97     74293
           1       0.00      0.25      0.00         4

    accuracy                           0.94     74297
   macro avg       0.50      0.60      0.49     74297
weighted avg       1.00      0.94      0.97     74297

